In [1]:
import pandas as pd
import numpy as np


In [7]:
# 데이터 로드
df = pd.read_csv("mock_data_2.csv").copy()

# 순서와 크소어 소드 매핑
use_frequency_map = {
    "rare": 1,
    "monthly": 2,
    "weekly": 3,
    "frequent": 4
}

last_use_recency_map = {
    ">30d": 1,
    "7-30d": 2,
    "1-7d": 3,
    "<1d": 4,
}

df["use_frequency_score"] = df["use_frequency"].map(use_frequency_map)
df["last_use_recency_score"] = df["last_use_recency"].map(last_use_recency_map)

df["log_monthly_cost"] = np.log1p(df["monthly_cost"]) # 오른쪽 꼬리 완화
df["monthly_cost_z"] = (df["monthly_cost"] - df["monthly_cost"].mean()) / df["monthly_cost"].std(ddof=0)

# value_gap: "사용가치 - 비용부담" 관점
df["value_gap"] = (
    df["use_frequency_score"]
    + df['last_use_recency_score']
    + df['perceived_necessity']
    - df['cost_burden']
)

# rebuy_satisfaction_gap:
# perceived_necessity 대체 프록시로 사용
if "satisfaction" in df.columns:
    df["rebuy_satisfaction_gap"] = df["monthly_cost"] - df["satisfaction"]
else:
    df["rebuy_satisfaction_gap"] = df["monthly_cost"] - df["perceived_necessity"]

# cost_to_necessity_ratio:
df["cost_to_necessity_ratio"] = df["monthly_cost"] * (df["perceived_necessity"] + 1e-6)

df["cost_burden_x_replacement"] = df["cost_burden"] * df["replacement_available"]
df["necessity_x_recency"] = df["perceived_necessity"] * df["last_use_recency_score"]
df["frequency_x_rebuy"] = df["use_frequency_score"] * df["would_rebuy"]

high_cost_threshold = df["monthly_cost"].quantile(0.75)
df["is_high_cost"] = (df["monthly_cost"] > high_cost_threshold).astype(int)

# 이탈 신호 플래그(규칙 기반)
df["has_churn_signal"] = (
    (df["use_frequency_score"] <= 2)
    & (df["last_use_recency_score"] <= 2)
    & (df["cost_burden"] >= 4)
).astype(int)

# 6) 확인
new_cols = [
    "use_frequency_score",
    "last_use_recency_score",
    "value_gap",
    "rebuy_satisfaction_gap",
    "cost_to_necessity_ratio",
    "log_monthly_cost",
    "monthly_cost_z",
    "cost_burden_x_replacement",
    "necessity_x_recency",
    "frequency_x_rebuy",
    "is_high_cost",
    "has_churn_signal",
]
display(df[new_cols].head())
display(df[new_cols].describe().T)

,use_frequency_score,last_use_recency_score,value_gap,rebuy_satisfaction_gap,cost_to_necessity_ratio,log_monthly_cost,monthly_cost_z,cost_burden_x_replacement,necessity_x_recency,frequency_x_rebuy,is_high_cost,has_churn_signal
0,3,4,9,10431,31302.010434,9.252921,-0.557620,0,12,12,0,0
1,3,4,10,9502,38024.009506,9.159784,-0.660058,0,16,15,0,0
2,2,1,0,25875,25876.025876,10.161110,1.146951,0,1,6,1,1
3,2,1,1,31877,95640.031880,10.369766,1.809705,5,3,2,1,1
4,4,4,11,9904,39632.009908,9.201199,-0.615683,1,16,16,0,0


,count,mean,std,min,25%,50%,75%,max
use_frequency_score,20000.0,2.711050e+00,1.111764,1.000000,2.000000,3.000000,4.000000,4.000000
last_use_recency_score,20000.0,2.628600e+00,1.206497,1.000000,1.000000,3.000000,4.000000,4.000000
value_gap,20000.0,6.690300e+00,3.099269,-2.000000,4.000000,7.000000,9.000000,12.000000
rebuy_satisfaction_gap,20000.0,1.548223e+04,9059.342645,2895.000000,8572.750000,12879.500000,21159.000000,58997.000000
cost_to_necessity_ratio,20000.0,5.230157e+04,36975.951074,2900.002900,24822.756723,41862.013954,71198.266910,257930.051586
log_monthly_cost,20000.0,9.472456e+00,0.611147,7.972811,9.056839,9.463703,9.959962,10.985310
monthly_cost_z,20000.0,2.806644e-17,1.000025,-1.389264,-0.762717,-0.287342,0.626484,4.803357
cost_burden_x_replacement,20000.0,1.014650e+00,1.265866,0.000000,0.000000,1.000000,2.000000,5.000000
necessity_x_recency,20000.0,9.305050e+00,5.855255,1.000000,4.000000,8.000000,15.000000,20.000000
frequency_x_rebuy,20000.0,8.229700e+00,5.457603,1.000000,3.000000,8.000000,12.000000,20.000000


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OntHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, auc_roc_score,
    confusion_matrix, classification_report
)

SyntaxError: invalid syntax (3021312598.py, line 6)